# One backbone, three ways of finetuning it

The paper finetunes every model the CultureLLM way: supervised finetuning toward one
target answer per question. A reviewer put it that the trade the paper reports, average
answers moving while the variation between groups is lost, may belong to that
single-answer objective and not to cultural finetuning as such. This notebook is the
counter-test. It scores two distribution-trained variants of one backbone beside the
as-released, German and Mexican-Spanish variants of the same backbone, with the same
`group_fidelity` on the same prompts and cells: one trained to match country-level WVS
answer distributions with the first-token KL objective of Cao et al. (arXiv:2502.07068),
and one trained to match the answer distributions of demographic subgroups with the
SubPOP recipe of Suh et al. (arXiv:2502.16761).

In the code the two are the `global` and `subpop` arms and read as *distribution-matched*
and *subgroup-matched*. Both are kept out of `population_fidelity_evaluation.ipynb` on
purpose: they exist for one backbone, so folding them into the shared tables would move
every count the paper reports (164 combinations, 4,440 subgroup scores, 19 conditions)
for controls that are not part of the design. Everything here is written to its own
files, all named `population_fidelity_objectives_*`, and drawn into
`figures/fidelity/objectives/`, so the whole comparison is removed by deleting this
notebook, `culture.fidelity_objectives`, its test, those tables and that folder.

`OBJECTIVE_MODELS` names the backbones compared and `DISTRIBUTION_ARMS` the
distribution-trained variants. A second backbone joins by adding its key to the tuple.

Every replicate is scored apart, each finetuned variant against its own as-released run
on the cells that pair retains, which is how the paper scores its repeated runs and what
lets the first replicate reproduce the published tables to the last digit.


In [1]:
from culture.fidelity import GROUPS_TABLE, PAIRED_TABLE, cross_check_cells
from culture.fidelity_objectives import (
    OBJECTIVES_BASELINE_TABLE,
    OBJECTIVES_GROUPS_TABLE,
    OBJECTIVES_PAIRED_TABLE,
    OBJECTIVES_REPLICATES_TABLE,
    OBJECTIVES_SUMMARY_TABLE,
    build_objectives,
    check_objectives,
    objective_baseline,
    objective_plates,
    objective_runs,
    objective_summary,
    replicate_spread,
)
from culture.tables import read_csv
from machine_bias_reproduction.io_utils import write_csv

## Build

In [2]:
runs = objective_runs()
cells, groups, paired = build_objectives(runs=runs)
print(len(groups), "group rows and", len(paired), "paired rows over", groups["source"].nunique(), "runs")

2356 group rows and 1488 paired rows over 14 runs


## Reconciling against the runs and the published tables

Two checks before anything is summarised. The per-cell distances recomputed here are
the ones each run wrote at analysis time. And the three variants the paper already
reports come out of this build exactly as they stand in
`population_fidelity_groups.csv` and `population_fidelity_paired.csv`, row for row, on
the same cells, so the two distribution-trained variants are read on the paper's own
scale.


In [3]:
published_groups, published_paired = read_csv(GROUPS_TABLE), read_csv(PAIRED_TABLE)
assert published_groups is not None and published_paired is not None
matched = check_objectives(groups, paired, published_groups, published_paired)
print(cross_check_cells(cells), "runs match their split outputs;", matched, "rows match the published tables")

76 runs match their split outputs; 1240 rows match the published tables


## Accuracy against the survey center

The same survey-center baseline as `population_fidelity_baseline.ipynb`, on the first
replicate of the five variants: does matching distributions lift cell-level accuracy
above a prediction that ignores demographics?


In [4]:
baseline = objective_baseline(runs=runs)
print(len(baseline), "baseline rows")

1240 baseline rows


## The tables

`population_fidelity_objectives_summary.csv` holds one row per scope and variant, read
on the first replicate as the paper reads every score. It follows the paper's two
conventions: the level of a score across the question-by-mode combinations is its
median, beside its minimum and maximum, and what finetuning changed is the mean paired
change from the as-released model, beside the count of combinations that moved up, down
or not at all. It also counts how often structure is the smallest term and how often
accuracy beats the survey center. Scope `population` is the eight pooled combinations;
`subgroups` is every demographic level of them; `Germany` and `Mexico` score every
variant on the cells of that country alone, which is where the German and the
Mexican-Spanish variant were finetuned to do well.

`population_fidelity_objectives_replicates.csv` holds the standard deviation of each
score across the full-answer runs of a variant, per question, closed by an `all` row
that pools the four questions as the root mean square.

In [5]:
summary = objective_summary(groups, paired, baseline)
spread = replicate_spread(groups)
write_csv(groups, OBJECTIVES_GROUPS_TABLE)
write_csv(paired, OBJECTIVES_PAIRED_TABLE)
write_csv(baseline, OBJECTIVES_BASELINE_TABLE)
write_csv(summary, OBJECTIVES_SUMMARY_TABLE)
write_csv(spread, OBJECTIVES_REPLICATES_TABLE)
print(len(summary), "summary rows;", len(spread), "spread rows")

20 summary rows; 25 spread rows


## The comparison

In [6]:
pooled = summary[summary["scope"].eq("population")].set_index("arm")
medians = [f"{score}_median" for score in ("score_accuracy", "adaptability_ratio", "score_structure", "pfs", "score_center")]
print(pooled[medians].round(3).T.to_string())
for country in ("Germany", "Mexico"):
    inside = summary[summary["scope"].eq(country)].set_index("arm")
    print(f"\n{country} cells only")
    print(inside[medians].round(3).T.to_string())


arm                         base  german  spanish-mx  global  subpop
score_accuracy_median      0.823   0.797       0.816   0.848   0.815
adaptability_ratio_median  0.561   0.131       0.312   0.549   0.263
score_structure_median     0.055   0.024       0.027   0.082   0.032
pfs_median                 0.259   0.164       0.093   0.330   0.167
score_center_median        0.842   0.827       0.841   0.883   0.843

Germany cells only
arm                         base  german  spanish-mx  global  subpop
score_accuracy_median      0.856   0.818       0.827   0.871   0.800
adaptability_ratio_median  0.733   0.164       0.428   0.777   0.327
score_structure_median     0.035   0.022       0.047   0.037   0.044
pfs_median                 0.237   0.145       0.203   0.291   0.188
score_center_median        0.870   0.847       0.853   0.892   0.811

Mexico cells only
arm                         base  german  spanish-mx  global  subpop
score_accuracy_median      0.696   0.743       0.808   0.779   0

## The plates

Three plates per question and elicitation mode. The first stacks the component profile
three ways in one figure, which is how the supervisor asked to read it: every variant on
all retained cells, then the variants that belong on the German cells alone (as released,
German, and the two distribution-trained variants, which condition on no single
country), then the same on the Mexican cells with the Mexican-Spanish variant in place of
the German one. Rows keep one order in every block so a variant can be followed down the
figure. The other two are drawn as the paper draws them for the whole sweep: the
component profile of the five variants ordered by PFS, and the change of each finetuned
variant from the as-released model, PFS across and center alignment up, so the top-left
corner holds what center alignment alone would count as an improvement. The `germany/`
and `mexico/` folders hold those two plates scored on the cells of that country alone.


In [7]:
files = objective_plates(groups, paired)
print(len(files), "files")

112 files


## Reading the comparison

The distribution-matched variant is trained on country-level answer distributions,
which is the kind of target center alignment measures, and it succeeds: its median
center alignment is the highest of the five variants and it rises in seven of the eight
pooled combinations. PFS rises in three of them, both social-trust combinations among
them, falls in three and stays at zero in two. Read on the center alone, the variant
improved almost everywhere; PFS says where that improvement came with fidelity and where
it did not.

Read the adaptability ratio first. Single-answer finetuning flattens the model across
cells: the median ratio falls from the as-released value toward zero under both the
German and the Mexican-Spanish variant. The distribution-matched variant keeps it where
the as-released model had it. On this backbone the loss of between-group variation
belongs to the single-answer objective. Center alignment cannot tell the two apart.
Trust under NTP is the clean case: the Mexican-Spanish and the distribution-matched
variant reach the same center alignment and a PFS of 0.000 against 0.425.

The subgroup-matched variant is the one trained with information about groups, and it
behaves like the single-answer variants on the properties center alignment cannot see:
its median center alignment sits on the as-released value, its adaptability ratio falls
to 0.263, structure binds in all eight combinations and PFS falls in six. Its two gains
are religious attendance under both modes, the item SubPOP-Train holds eight times; on
happiness, which it never saw, PFS falls by a fifth. Under happiness with FA it keeps the
distribution-matched variant's center alignment (0.952 against 0.953) with a PFS of 0.250
against 0.437, the same disagreement between the two readings as trust under NTP shows
for the Mexican-Spanish variant.

Then read structure. It stays the smallest term in every pooled combination under every
objective. Matching country-level distributions moves the average answer and keeps the
spread, matching subgroup-level distributions on one country's items moves neither
toward the survey, and outside social trust neither recovers which groups differ from
which. No variant beats the survey center on accuracy either.

Scored on its own target country the single-answer objective does no better, which is
what the stacked plate shows. The German variant loses center alignment and PFS inside
Germany in six of the eight combinations. The Mexican-Spanish variant gains center
alignment inside Mexico, up to 0.966 on trust under NTP, and its PFS there rises in two
combinations, falls in four and stays at zero in two, because it flattens the variation
among Mexican cells. The subgroup-matched variant, trained on United States subgroups,
loses inside Germany and reaches the highest center alignment and accuracy inside
Mexico, with PFS up in five of eight; its gains there sit on the two items its training
data hold, and the Mexican cells are compressed all the same.

Three things bound the reading. The distribution-matched variant saw the happiness and
social-trust items during training for four of the five countries scored here, and
religious attendance steered its checkpoint selection, so only the left-right item is
unseen; SubPOP-Train holds eight religious-attendance items and one trust item for
United States subgroups, so neither target country of the single-answer variants is
represented in it. And both are one backbone, the smallest in the paper.
